# Exploratory analysis of the dataset

What the point-in-time dataset looks like before modelling. Five questions:

1. Who is in the data, and how many players are eligible at each cutoff?
2. How often do eligible players stop playing, by cutoff, region and tier?
3. What do the features look like, and how much is missing?
4. How strongly is each feature related to churn?
5. Is there signal beyond recency? The classical models only edge out the rule "days since the
   last game" (README, *First results*), so this is the question that decides what to try next.

The notebook reads only `data/processed/<dataset>/examples.parquet`, written by `lolret build`,
never raw API responses. The player ID (`puuid`) is dropped when the data is loaded, so no cell
can show it. Outputs are stripped on commit, so the findings at the end are written out.

**Run it:** `uv sync --group notebooks`, build the dataset (`uv run lolret build -c
configs/collect.yaml`), then open this notebook in Jupyter or VS Code with the project's
`.venv` as the kernel.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from sklearn.metrics import roc_auc_score

from lolret.config import get_settings
from lolret.model.config import load_train_config
from lolret.model.data import FEATURES
from lolret.model.train import examples_path

# The notebook can run from notebooks/ or from the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = load_train_config(ROOT / "configs" / "train.yaml").dataset
data_dir = get_settings().data_dir
SOURCE = examples_path(data_dir if data_dir.is_absolute() else ROOT / data_dir, DATASET)

TIERS = ["IRON", "BRONZE", "SILVER", "GOLD", "PLATINUM", "EMERALD", "DIAMOND"]
REGIONS = ["na1", "euw1", "kr"]
RECENCY_BINS = [-0.01, 1, 3, 7, 14, 28]
RECENCY_GROUPS = ["<1 day", "1-3 days", "3-7 days", "7-14 days", "14-28 days"]

In [ ]:
def pct(frame, digits=1):
    """Shares as rounded percentages, for display."""
    return (frame * 100).round(digits)


def auc_both_ways(y, x):
    """ROC-AUC of one feature whichever way it points, and the direction that means churn."""
    keep = x.notna()
    auc = roc_auc_score(y[keep], x[keep])
    return max(auc, 1 - auc), "higher" if auc >= 0.5 else "lower"


def deciles(values):
    """Decile bins. A feature that is mostly zero gets zero plus quartiles of the rest."""
    bins = pd.qcut(values, 10, duplicates="drop")
    if bins.cat.categories.size >= 3:
        return bins
    edges = [-np.inf, 0, *values[values > 0].quantile([0.25, 0.5, 0.75]), np.inf]
    return pd.cut(values, edges, duplicates="drop")


# Chart style: light surface, hairline grid, quiet axes, one blue for single-series charts.
SURFACE, INK, INK_2, MUTED, GRID, BASELINE = (
    "#fcfcfb",
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#c3c2b7",
)
BLUE = "#2a78d6"
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
        "font.size": 9,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 9,
        "axes.labelcolor": INK_2,
        "axes.edgecolor": BASELINE,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 1,
        "xtick.color": BASELINE,
        "ytick.color": BASELINE,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)

In [ ]:
# puuid is excluded here, so no cell below can show a player identifier.
rows = duckdb.execute("SELECT * EXCLUDE (puuid) FROM read_parquet(?)", [str(SOURCE)]).df()
rows["cutoff"] = rows.cutoff.dt.strftime("%Y-%m-%d")
cutoffs = sorted(rows.cutoff.unique())
# Every player has one row per cutoff, so the rows at one cutoff are the players.
players = rows[rows.cutoff == cutoffs[0]]

examples = rows[rows.eligible & rows.churned.notna()].copy()
examples["churned"] = examples.churned.astype(int)
# No match features means a game was running at the cutoff: the recency rule counts that as 0.
examples["recency"] = pd.cut(
    examples.days_since_last_game.fillna(0), RECENCY_BINS, labels=RECENCY_GROUPS
)
CHURN_RATE = examples.churned.mean()
print(
    f"{len(players):,} players x {len(cutoffs)} cutoffs = {len(rows):,} rows. "
    f"{len(examples):,} eligible with a known label, {CHURN_RATE:.1%} of them churned."
)

## 1. Population

Players were sampled from random ladder pages, the same number per division in each region
(ADR 0005), so tiers are balanced by design, not by how common they are in the game. Tier and
division are measured at collection time, after every cutoff, so they are used here to describe
the data and never as features (ADR 0006).

In [ ]:
by_tier = pd.crosstab(players.tier, players.platform, margins=True, margins_name="all")
by_tier.reindex(index=[*TIERS, "all"], columns=[*REGIONS, "all"])

A player is **eligible** at a cutoff if they started at least one game, in any queue, in the
28 days before it. Share of sampled players who are eligible (%):

In [ ]:
eligible = rows.pivot_table(index="cutoff", columns="platform", values="eligible")
pct(eligible[REGIONS].assign(all=rows.groupby("cutoff").eligible.mean()))

## 2. Churn

**Churned** means no game at all in the 28 days after the cutoff. Churn rate among eligible
examples (%), by cutoff and region. Each cell holds about 1,300 examples, so differences of
2 points or less between cells are within noise.

In [ ]:
churn = examples.pivot_table(index="cutoff", columns="platform", values="churned")[REGIONS]
churn = churn.assign(all=examples.groupby("cutoff").churned.mean())
churn.loc["all"] = [*examples.groupby("platform").churned.mean()[REGIONS], CHURN_RATE]
pct(churn)

In [ ]:
by_tier_churn = examples.groupby("tier").churned.mean().reindex(TIERS)

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.barh(TIERS, by_tier_churn * 100, height=0.55, color=BLUE)
ax.axvline(CHURN_RATE * 100, color=MUTED, linewidth=1, zorder=0)
ax.text(CHURN_RATE * 100, len(TIERS) - 0.45, f" all tiers {CHURN_RATE:.1%}", color=INK_2)
for y, rate in enumerate(by_tier_churn):
    ax.text(rate * 100 + 0.4, y, f"{rate:.1%}", va="center", color=INK_2)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("Churn rate among eligible examples (%)")
ax.set_title("Churn by tier (tier measured at collection time)", loc="left")
plt.show()

In [ ]:
by_tier_region = examples.pivot_table(index="tier", columns="platform", values="churned")
pct(by_tier_region.reindex(index=TIERS, columns=REGIONS))

## 3. Features

Definitions are in `docs/features.md`. Missing share (%) and the spread of each feature among
eligible examples:

In [ ]:
summary = examples[list(FEATURES)].astype(float).describe(percentiles=[0.05, 0.5, 0.95]).T
summary.insert(0, "missing %", pct(examples[list(FEATURES)].isna().mean(), 2))
summary.drop(columns="count").round(2)

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(11, 12))
for ax, feature in zip(axes.flat, FEATURES, strict=False):
    values = examples[feature].dropna().astype(float)
    low, high = values.quantile([0.01, 0.99])
    ax.hist(values.clip(low, high), bins=30, color=BLUE, edgecolor=SURFACE, linewidth=0.5)
    ax.set_title(feature, loc="left")
    ax.set_yticks([])
    ax.grid(False)
for ax in axes.flat[len(FEATURES) :]:
    ax.set_visible(False)
fig.suptitle(
    "Feature distributions, eligible examples (clipped to the 1st-99th percentile)",
    x=0.01,
    ha="left",
    color=INK,
)
fig.tight_layout()
plt.show()

A few values look odd. They are counted here so their size is clear:

In [ ]:
pd.Series(
    {
        "gap_hours_median < 0 (games overlap in the timestamps)": (
            examples.gap_hours_median < 0
        ).sum(),
        "duration_min_avg < 5 (almost only remakes)": (examples.duration_min_avg < 5).sum(),
        "recent_games < 20 (fewer than 20 games in the history)": (
            examples.recent_games < 20
        ).sum(),
    },
    name="examples",
)

## 4. Each feature against churn

ROC-AUC of each feature on its own, over all eligible examples: 0.5 means no signal, 1.0 a
perfect ranking. It is computed whichever way the feature points, and the last column says
which direction means more churn.

In [ ]:
signal = pd.DataFrame(
    [(feature, *auc_both_ways(examples.churned, examples[feature])) for feature in FEATURES],
    columns=["feature", "auc", "more churn when value is"],
).set_index("feature")
signal = signal.sort_values("auc", ascending=False)
signal.round(3)

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(11, 13), sharey=True)
for ax, feature in zip(axes.flat, signal.index, strict=False):
    data = examples[[feature, "churned"]].dropna()
    bins = deciles(data[feature].astype(float))
    by_bin = data.groupby(bins, observed=True).agg(
        value=(feature, "median"), rate=("churned", "mean")
    )
    ax.axhline(CHURN_RATE * 100, color=MUTED, linewidth=1)
    ax.plot(
        by_bin.value,
        by_bin.rate * 100,
        color=BLUE,
        marker="o",
        markersize=6,
        markeredgecolor=SURFACE,
        markeredgewidth=1.5,
    )
    ax.set_title(f"{feature}  (AUC {signal.loc[feature, 'auc']:.2f})", loc="left")
for ax in axes.flat[len(FEATURES) :]:
    ax.set_visible(False)
fig.suptitle(
    "Churn rate (%) by decile of each feature; grey line = overall rate",
    x=0.01,
    ha="left",
    color=INK,
)
fig.tight_layout()
plt.show()

## 5. Signal beyond recency

Split the examples by days since the last game, the recency rule's only input. Churn climbs
steeply from group to group, and most churners sit in the last two groups:

In [ ]:
groups = examples.groupby("recency", observed=True).churned.agg(["size", "mean", "sum"])
pd.DataFrame(
    {
        "examples": groups["size"],
        "churn rate %": pct(groups["mean"]),
        "share of all churners %": pct(groups["sum"] / groups["sum"].sum()),
    }
)

Within each group, recency is roughly constant, so a feature that still ranks churners there
adds information the rule doesn't have. ROC-AUC of each feature inside each group, whichever way
it points. Grey means the feature is constant in that group: for example, a player whose last
game was 7 or more days ago has no games in the last 7 days.

In [ ]:
within = pd.DataFrame(
    {
        group: {
            feature: auc_both_ways(g.churned, g[feature])[0] if g[feature].nunique() > 1 else np.nan
            for feature in FEATURES
        }
        for group, g in examples.groupby("recency", observed=True)
    }
).reindex(signal.index)

steps = np.linspace(0.5, 0.78, len(BLUE_RAMP) + 1)
cmap = ListedColormap(BLUE_RAMP).with_extremes(bad=GRID)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
image = ax.imshow(
    within.to_numpy(dtype=float), cmap=cmap, norm=BoundaryNorm(steps, cmap.N), aspect="auto"
)
ax.set_xticks(range(len(within.columns)), within.columns)
ax.set_yticks(range(len(within.index)), within.index)
ax.xaxis.tick_top()
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(image, ax=ax, shrink=0.5, label="AUC inside the group (0.5 = no signal)")
ax.set_title("Days since the last game", loc="left", pad=24)
plt.show()

within.round(3)

The same thing in plain rates for the strongest feature in the high-risk groups: churn by
number of active weeks out of the last 8 (%).

In [ ]:
high_risk = examples[examples.recency.isin(RECENCY_GROUPS[3:])]
pct(
    high_risk.pivot_table(
        index="active_weeks_8w", columns="recency", values="churned", observed=True
    )
)

## Findings (run `v1`, 2026-10-24)

- **Population.** 6,048 players, 288 per tier in each region, by design. About two thirds are
  eligible at a cutoff, slightly fewer at later ones (66.2%, 64.7%, 62.2%), which leaves 11,678
  eligible examples.
- **Churn is stable over time.** 17.1%, 18.9% and 17.7% at the three cutoffs, and 16.8-18.6% by
  region. The test cutoff isn't unusual, so calibration fitted on the earlier cutoffs carries
  over, as the model card shows. The 37% seen on the first 122 players was a partial-data effect.
- **Churn depends on tier.** 8% in Diamond against 23-25% in Silver and Bronze, with Iron lower
  (17%). Tier is measured after the cutoff, so it can't be a feature, but results should be
  broken down by tier later.
- **The features are almost complete.** Only `main_role_share` (2.4%) and `gap_hours_median`
  (0.9%) have gaps. Counts are strongly right-skewed: a median of 4 games in the last week, a
  95th percentile of 38. 17 examples have a slightly negative `gap_hours_median` (down to -10
  minutes), where two games overlap in the timestamps. Too few to matter.
- **How much people play predicts churn; how they play barely does.** Days since the last game
  (AUC 0.80), games in the last 7 days (0.79) and active weeks (0.75) lead. Every in-game and
  play-style feature (win rate, KDA, deaths, surrenders, remakes, champion diversity, roles)
  sits between 0.50 and 0.55.
- **Recency splits the population sharply.** Churn goes from 3% for players whose last game was
  less than a day before the cutoff to 50% for those at 14-28 days, and 69% of all churners are
  in the 7-28 day groups.
- **The signal left beyond recency is mostly where churners are rare.** Among players who played
  in the last 3 days, activity volume still separates churners well (AUC 0.68-0.77 inside the
  group). In the 7-28 day groups, where most churners are, the best feature (active weeks)
  reaches only 0.57-0.60, and in-game features stay near 0.51-0.53. PR-AUC is driven by the top
  of the ranking, which is exactly those groups, so the models can only edge out the rule.

**What to try next.** More in-game statistics are unlikely to help. What's missing is anything
that tells a break from quitting among players who already stopped for a week or more: their
history of earlier breaks and returns, activity over a longer window than 8 weeks, or account
age. The GRU planned for Phase 4b sees only the last ~20 matches, so it may hit the same limit.